This notebook contains code to create a csv of all pulsar measurements with corrected fluxes instead of the raw fluxes detected by ASKAP.
It should be run after 'get_all_control_measurements.ipynb'

IN: 'all_candidate_control_measurements.csv', 'all_pulsar_measurements.csv', 'paper_dfv2.csv'

OUT: 'all_pulsar_measurements_corrected.csv'

In [2]:
import numpy as np

import pandas as pd

from astropy.coordinates import SkyCoord

#the two below imports are to help stop vasttools messages from overcrowding the console
import logging
logging.basicConfig(level=201)

import warnings
warnings.filterwarnings('ignore')

In [9]:
#reading in necessary premade csvs from previous notebooks
psrs = pd.read_csv('paper_dfv2.csv')

controls_measurements = pd.read_csv('all_candidate_control_measurements.csv')
raw_pulsar_meas = pd.read_csv('all_pulsar_measurements.csv')

#editing dataframe to remove superfluous columns. this may need to be changed depending on your data
raw_pulsar_meas.rename(columns={raw_pulsar_meas.columns[0]: 'psr_name'}, inplace=True)
raw_pulsar_meas.drop(columns=raw_pulsar_meas.columns[1:3], inplace=True)

corrected_pulsar_meas = raw_pulsar_meas.copy()

In [10]:
#computing raw flux / control flux, the error in this measurement, and adding these values to a container
#some pulsars may not have viable control (for example PSR J1752-2806), so effort is made to skip these pulsars
#the debugging printout displays whether or not the pulsar has an acceptable control csource, and if so, which control source this is and how many corrected detections it produced
corrected_flux_series = pd.Series(dtype='Float64')
corrected_err_series = pd.Series(dtype='Float64')
unfilled_psrs = []
filled_psrs = []
for name in psrs['JNAME']:
    print(name)
    
    control_sources = controls_measurements[
        (controls_measurements['PSR_assoc']==name)
    ]
    
    for i in pd.unique(control_sources['control_number']):
        if (name in filled_psrs):
            #print("                skipped")
            continue
        
        print("        control number " + str(i))
        control_meas = control_sources[control_sources['control_number']==i]
        #if control_meas['epoch'].str.contains('3x').any():
        #    control_meas.where(~control_meas['epoch'].str.contains('3x'), '3', inplace=True)
        #if control_meas['epoch'].str.contains('5x').any():
        #    control_meas.where(~control_meas['epoch'].str.contains('5x'), '5', inplace=True)
        #if control_meas['epoch'].str.contains('10x').any():
        #    control_meas.where(~control_meas['epoch'].str.contains('10x'), '10', inplace=True)
        control_meas['epoch'] = control_meas['epoch'].apply(int)

        psr_meas = raw_pulsar_meas[raw_pulsar_meas['psr_name']==name]
        if ((psr_meas['epoch'].apply(type)==str).any()): #check the datatype in the epochs field for this particular pulsar
            if psr_meas['epoch'].str.contains('3x').any():
                psr_meas.replace('3x', '3', inplace=True)
            if psr_meas['epoch'].str.contains('5x').any():
                psr_meas.replace('5x', '5', inplace=True)
            if psr_meas['epoch'].str.contains('10x').any():
                psr_meas.replace('10x', '10', inplace=True)
            psr_meas['epoch'] = psr_meas['epoch'].apply(int)

        control_meas = control_meas[control_meas['detection']]
        psr_meas = psr_meas[psr_meas['detection']]
        num_psr_detections = psr_meas[psr_meas['detection']]['flux_peak'].count()

        epochs = np.intersect1d(control_meas['epoch'], psr_meas['epoch'])
        control_mask = (control_meas['epoch'].isin(epochs))
        psr_mask = (psr_meas['epoch'].isin(epochs))

        psr_meas = psr_meas[psr_mask]
        control_meas = control_meas[control_mask]
        
        #each these measurement lists now need to fulfill the following condition to produce satisfactory corrected fluxes:
            # must have either the same number of detections as the original pulsar measurements
            # OR more than 30 detections
        cond1 = (psr_meas['flux_peak'].count() >= num_psr_detections - 8)&(psr_meas['flux_peak'].count() > 1)
        #cond2 = psr_meas['flux_peak'].count() > 1
        cond3 = psr_meas['flux_peak'].count() > 30
        
        if ((i==pd.unique(control_sources['control_number'])[-1])&(not cond1)&(not cond3)): # check if at end of loop and unfilled
            unfilled_psrs.append((name, psr_meas['flux_peak'].count()))
            print("                pulsar unfilled")
        elif ((not cond1)&(not cond3)): # check if not at end of loop and unfilled
            print("                skipped")
            continue
        else:
            print("        " + name + " filled, with " + str(psr_meas['flux_peak'].count()) + " detections")
            filled_psrs.append(name)

        for i in epochs:
            if (psr_meas[psr_meas['epoch']==i].shape[0])>(control_meas[control_meas['epoch']==i].shape[0]):
                size = control_meas[control_meas['epoch']==i].shape[0]
                psr_meas.drop(psr_meas[psr_meas['epoch']==i].index[size:], inplace=True)
            elif (psr_meas[psr_meas['epoch']==i].shape[0])<(control_meas[control_meas['epoch']==i].shape[0]):
                size = psr_meas[psr_meas['epoch']==i].shape[0]
                control_meas.drop(control_meas[control_meas['epoch']==i].index[size:], inplace=True)
        control_meas = pd.DataFrame(data=control_meas.values, columns=control_meas.columns, index=psr_meas.index)

        flux_ratios = psr_meas['flux_peak'].div(control_meas['flux_peak'])
        errs = (psr_meas['rms_image'].div(psr_meas['flux_peak']).pow(2) + control_meas['rms_image'].div(control_meas['flux_peak']).pow(2)).pow(0.5).mul(flux_ratios)
        corrected_flux_series = pd.concat([corrected_flux_series, flux_ratios])
        corrected_err_series = pd.concat([corrected_err_series, errs])

J1644-4559
        control number 53
        J1644-4559 filled, with 41 detections
J1752-2806
        control number 8
                skipped
        control number 29
                skipped
        control number 18
                skipped
        control number 19
                skipped
        control number 44
                skipped
        control number 32
                skipped
        control number 36
                skipped
        control number 24
                skipped
        control number 14
                skipped
        control number 21
                skipped
        control number 16
                skipped
        control number 37
                skipped
        control number 22
                skipped
        control number 27
                skipped
        control number 12
                skipped
        control number 35
                skipped
        control number 38
                skipped
        control number 26
                skipped
       

                skipped
        control number 4
                skipped
        control number 17
                skipped
        control number 9
                skipped
        control number 18
                skipped
        control number 5
                skipped
        control number 3
                skipped
        control number 2
                skipped
        control number 6
                skipped
        control number 1
                skipped
        control number 7
                skipped
        control number 0
                skipped
        control number 23
                pulsar unfilled
J1811-2405
        control number 29
                skipped
        control number 70
        J1811-2405 filled, with 38 detections
J1539-5626
        control number 64
                skipped
        control number 63
                skipped
        control number 52
        J1539-5626 filled, with 41 detections
J1633-4453
        control number 72
        J1633-4453 fille

J1832-1021
        control number 31
                skipped
        control number 36
        J1832-1021 filled, with 39 detections
J1759-2922
        control number 26
        J1759-2922 filled, with 31 detections
J1825-1446
        control number 23
                skipped
        control number 16
                skipped
        control number 40
        J1825-1446 filled, with 42 detections
J1553-5456
        control number 67
        J1553-5456 filled, with 33 detections
J1801-3210
        control number 71
        J1801-3210 filled, with 35 detections
J1707-4341
        control number 60
        J1707-4341 filled, with 37 detections
J1826-1334
        control number 74
        J1826-1334 filled, with 42 detections
J1639-4359
        control number 68
        J1639-4359 filled, with 41 detections
J1810-2005
        control number 36
        J1810-2005 filled, with 35 detections
J1812-1718
        control number 80
        J1812-1718 filled, with 31 detections
J1843-1113
        c

In [12]:
#editing the dataframe for the corrected fluxes
corrected_psr_meas['flux_peak'] = corrected_flux_series
corrected_psr_meas['rms_image'] = corrected_err_series

In [14]:
#saving this to a csv
corrected_psr_meas.to_csv('all_pulsar_measurements_corrected.csv')